In [68]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler

In [69]:
# Reading the first dataset containing movies meta data as movie_metadata.csv
df1=pd.read_csv(r'D:/DataMining_coursework/movie_metadata.csv')
#Reading the second dataset containing the ratings of movies on netflix 
df2=pd.read_csv(r'D:/DataMining_coursework/Netflix_Dataset_Movie.csv')

In [70]:
# info about dataset before cleaning and stuff 
# Dataset 1
print("INFO OF DATASET ONE")
df1.info()
print("-" * 40)

# Dataset 2
print("INFO OF DATASET TWO")
df2.info()

INFO OF DATASET ONE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5024 non-null   object 
 1   director_name              4939 non-null   object 
 2   num_critic_for_reviews     4993 non-null   float64
 3   duration                   5028 non-null   float64
 4   director_facebook_likes    4939 non-null   float64
 5   actor_3_facebook_likes     5020 non-null   float64
 6   actor_2_name               5030 non-null   object 
 7   actor_1_facebook_likes     5036 non-null   float64
 8   gross                      4159 non-null   float64
 9   genres                     5043 non-null   object 
 10  actor_1_name               5036 non-null   object 
 11  movie_title                5043 non-null   object 
 12  num_voted_users            5043 non-null   int64  
 13  cast_total_facebook_likes  5

In [71]:
#displaying the firstdataset
df1.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [72]:
#displaying the second dataset 
df2.head()

,Movie_ID,Year,Name
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review
2,3,1997,Character
3,4,1994,Paula Abdul's Get Up & Dance
4,5,2004,The Rise and Fall of ECW


#### Identify and handle missing values and duplicate records.

In [73]:
# handling the null values 
print(df1.isnull().sum())
print(df2.isnull().sum())

# dropping null values 
'''
Instead of imputing the dataset with mean and modes we just drop the rows
that have NA values as imputing would result in the loss of uniqueness of each movie and 
in my opinion is harmful for downstream analysis tasks 
'''
df1=df1.dropna(axis=0)

#dropping duplicates in place if any present 
df1.drop_duplicates(inplace=True)
df2.drop_duplicates(inplace=True)

color                         19
director_name                104
num_critic_for_reviews        50
duration                      15
director_facebook_likes      104
actor_3_facebook_likes        23
actor_2_name                  13
actor_1_facebook_likes         7
gross                        884
genres                         0
actor_1_name                   7
movie_title                    0
num_voted_users                0
cast_total_facebook_likes      0
actor_3_name                  23
facenumber_in_poster          13
plot_keywords                153
movie_imdb_link                0
num_user_for_reviews          21
language                      14
country                        5
content_rating               303
budget                       492
title_year                   108
actor_2_facebook_likes        13
imdb_score                     0
aspect_ratio                 329
movie_facebook_likes           0
dtype: int64
Movie_ID    0
Year        0
Name        0
dtype: int64


In [74]:
# info about dataset after cleaning and stuff 
# Dataset 1
print("INFO OF DATASET ONE")
df1.info()
print("-" * 40)

# Dataset 2
print("INFO OF DATASET TWO")
df2.info()

INFO OF DATASET ONE
<class 'pandas.core.frame.DataFrame'>
Index: 3722 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      3722 non-null   object 
 1   director_name              3722 non-null   object 
 2   num_critic_for_reviews     3722 non-null   float64
 3   duration                   3722 non-null   float64
 4   director_facebook_likes    3722 non-null   float64
 5   actor_3_facebook_likes     3722 non-null   float64
 6   actor_2_name               3722 non-null   object 
 7   actor_1_facebook_likes     3722 non-null   float64
 8   gross                      3722 non-null   float64
 9   genres                     3722 non-null   object 
 10  actor_1_name               3722 non-null   object 
 11  movie_title                3722 non-null   object 
 12  num_voted_users            3722 non-null   int64  
 13  cast_total_facebook_likes  3722 n

#### Use the IQR method to identify and remove outliers.

In [75]:
df_cleaned_1=df1
# Select the column to clean
column = ['num_critic_for_reviews','gross','duration','director_facebook_likes','num_user_for_reviews','num_voted_users']

# Calculate Quantiles
for col in column:
    Q1 = df_cleaned_1[col].quantile(0.25)
    Q3 = df_cleaned_1[col].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Define Bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify Outliers (for reporting)
    outliers = df_cleaned_1[(df_cleaned_1[col] < lower_bound) | (df_cleaned_1[col] > upper_bound)]
    print(f"Number of outliers detected in {col}: {len(outliers)}")

    # Remove Outliers
    df_cleaned_1 = df_cleaned_1[(df_cleaned_1[col] >= lower_bound) & (df_cleaned_1[col] <= upper_bound)]

print(f"Final shape after removing outliers: {df_cleaned_1.shape}")

Number of outliers detected in num_critic_for_reviews: 148
Number of outliers detected in gross: 238
Number of outliers detected in duration: 102
Number of outliers detected in director_facebook_likes: 350
Number of outliers detected in num_user_for_reviews: 189
Number of outliers detected in num_voted_users: 165
Final shape after removing outliers: (2530, 28)


### Merging the dataset

In [76]:
# Remove leading/trailing whitespaces and convert to lowercase
df_cleaned_1['movie_title'] = df_cleaned_1['movie_title'].str.strip().str.lower()
df2['Name'] = df2['Name'].str.strip().str.lower()

# Now perform the merge
df_merged = pd.merge(df_cleaned_1, df2, left_on='movie_title', right_on='Name', how='inner')

In [77]:
# Drop duplicates based on the movie title column
df_merged = df_merged.drop_duplicates(subset=['movie_title'], keep='first')

print(f"Shape after dropping duplicates: {df_merged.shape}")

Shape after dropping duplicates: (1238, 31)


In [78]:
df_merged.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes,Movie_ID,Year,Name
0,Color,Robert Zemeckis,240.0,96.0,0.0,10000.0,Colin Firth,18000.0,137850096.0,Animation|Drama|Family|Fantasy,...,PG,200000000.0,2009.0,14000.0,6.8,2.35,0,4179,1951,a christmas carol
3,Color,James Bobin,218.0,113.0,33.0,11000.0,Alan Rickman,40000.0,76846624.0,Adventure|Family|Fantasy,...,PG,170000000.0,2016.0,25000.0,6.4,1.85,30000,9641,1998,alice through the looking glass
4,Color,Robert Zemeckis,287.0,115.0,0.0,964.0,Anthony Hopkins,18000.0,82161969.0,Action|Adventure|Animation|Fantasy,...,PG-13,150000000.0,2007.0,12000.0,6.3,2.35,3000,7442,1999,beowulf
5,Color,Dean Parisot,135.0,90.0,23.0,233.0,Richard Burgi,957.0,110332737.0,Comedy|Crime,...,PG-13,100000000.0,2005.0,550.0,6.1,2.35,2000,1893,1977,fun with dick and jane
6,Color,Breck Eisner,163.0,124.0,42.0,848.0,Rainn Wilson,11000.0,68642452.0,Action|Adventure|Comedy|Thriller,...,PG-13,130000000.0,2005.0,973.0,6.0,2.35,0,5683,1943,sahara


In [79]:
scaler_std = StandardScaler()
# Apply Standardization (Mean=0, Std=1)
df_merged = scaler_std.fit_transform(df_merged.select_dtypes(include='number'))
df_merged_normalized = pd.DataFrame(df_merged)

In [80]:
df_merged_normalized.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,2.674537,-0.593676,-0.786447,9.422627,1.416395,3.417107,0.855391,3.581640,-0.799557,0.387010,2.268867,1.297266,4.450288,0.610277,1.137231,-0.299675,-0.902873,-4.037904
1,2.276031,0.522503,-0.500765,10.416679,3.886911,1.451684,-0.538426,6.381793,-0.170281,-0.497388,1.865796,2.142791,8.300852,0.204627,-0.708604,7.302003,0.190797,0.206432
2,3.525890,0.653818,-0.786447,0.440372,1.416395,1.622935,2.741488,2.059570,-0.170281,2.305705,1.597082,1.055687,3.750185,0.103215,1.137231,0.460493,-0.249515,0.296737
3,0.772577,-0.987622,-0.587336,-0.286280,-0.497468,2.530547,1.884129,-0.529922,-0.170281,0.454465,0.925297,0.814109,-0.257902,-0.099610,1.137231,0.207104,-1.360605,-1.689973
4,1.279766,1.244736,-0.422852,0.325062,0.630322,1.187361,0.987142,0.441466,1.088272,1.990919,1.328368,0.814109,-0.109830,-0.201023,1.137231,-0.299675,-0.601724,-4.760344
